# Notebook 3 — Inference
Load LoRA + ControlNet, give it a mask, generate realistic spleen ultrasound images.

## Cell 1 — Mount Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
pip install -q diffusers accelerate transformers xformers
echo "Done"

## Cell 2 — Load all three models

We load three things together:
1. **ControlNet weights** (from Notebook 2) — tells the model to follow the mask shape
2. **SD 1.5 pipeline** — the base generative model
3. **LoRA weights** (from Notebook 1) — adds ultrasound texture on top of SD 1.5

LoRA is applied at inference time — no need to merge files.

In [ ]:
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
import torch

BASE = "/content/drive/MyDrive/spleen_project"

print("Loading ControlNet...")
controlnet = ControlNetModel.from_pretrained(
    f"{BASE}/controlnet/output",
    torch_dtype=torch.float16
)

print("Loading SD 1.5 pipeline...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

print("Loading LoRA weights...")
pipe.load_lora_weights(f"{BASE}/lora/output/spleen_lora.safetensors")
pipe.fuse_lora(lora_scale=0.8)  # 0.8 = strong texture; lower = subtler

# UniPC scheduler — faster convergence than default DDPM
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_xformers_memory_efficient_attention()

print("Ready.")

## Cell 3 — Generate images

Output layout: **mask | original US | synthetic 1 | synthetic 2 | synthetic 3**

`controlnet_conditioning_scale` controls how strictly output follows the mask:
- `0.5` = loose (more creative)
- `0.8` = balanced (recommended)
- `1.0` = strict (very close to mask shape)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os, torch

BASE     = "/content/drive/MyDrive/spleen_project"
MASK_DIR = f"{BASE}/controlnet/data/conditioning_images"
IMG_DIR  = f"{BASE}/controlnet/data/images"
OUT_DIR  = f"{BASE}/inference/results"
os.makedirs(OUT_DIR, exist_ok=True)

# Change this to test different masks
FNAME = "ct1-1.png"

mask     = Image.open(os.path.join(MASK_DIR, FNAME)).convert("RGB").resize((512, 512))
original = Image.open(os.path.join(IMG_DIR,  FNAME)).convert("RGB").resize((512, 512))

# Three seeds = three diverse but realistic outputs
SEEDS   = [42, 123, 777]
results = []

for seed in SEEDS:
    generator = torch.Generator("cuda").manual_seed(seed)
    out = pipe(
        prompt="ultrasound image",
        image=mask,
        num_inference_steps=30,           # 30 steps = fast + good quality
        guidance_scale=7.5,               # how strongly to follow the text prompt
        controlnet_conditioning_scale=0.8,
        generator=generator,
    ).images[0]
    results.append(out)
    out.save(os.path.join(OUT_DIR, f"result_seed{seed}.png"))
    print(f"Generated seed={seed}")

# Display: mask | original | synthetic x3
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(mask,     cmap="gray"); axes[0].set_title("Input mask",  fontsize=11)
axes[1].imshow(original, cmap="gray"); axes[1].set_title("Original US", fontsize=11)
for i, img in enumerate(results):
    axes[i+2].imshow(img, cmap="gray")
    axes[i+2].set_title(f"Synthetic {i+1}", fontsize=11)
for ax in axes:
    ax.axis("off")

plt.suptitle(f"File: {FNAME}", fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved to", OUT_DIR)